In [8]:
import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.tools import InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated
import requests

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model='llama-3.1-8b-instant'
)

In [3]:
# tool create

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate


In [4]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency': 'BDT'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782000001,
 'time_last_update_utc': 'Sun, 21 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782086401,
 'time_next_update_utc': 'Mon, 22 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'BDT',
 'conversion_rate': 122.5169}

In [5]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 122.5169})

1225.169

In [6]:
# Tool Binding

model_with_tools = model.bind_tools([get_conversion_factor, convert])

In [19]:
messages = [HumanMessage('What is the conversion factor between BDT and USD, and based on that can you convert 10 bdt to usd')]

In [20]:
ai_message = model_with_tools.invoke(messages)

messages.append(ai_message)

In [21]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'BDT', 'target_currency': 'USD'},
  'id': 'vse4m5vfd',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_factor': 0.0124},
  'id': 'av1r2mstz',
  'type': 'tool_call'}]

In [22]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
        
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)



In [23]:
messages

[HumanMessage(content='What is the conversion factor between BDT and USD, and based on that can you convert 10 bdt to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vse4m5vfd', 'function': {'arguments': '{"base_currency":"BDT","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'av1r2mstz', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.0124}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 350, 'total_tokens': 398, 'completion_time': 0.084630927, 'completion_tokens_details': None, 'prompt_time': 0.02416345, 'prompt_tokens_details': None, 'queue_time': 0.0521448, 'total_time': 0.108794377}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

In [25]:
model_with_tools.invoke(messages).content

'The conversion factor between BDT and USD is 0.008162, and 10 BDT is equivalent to 0.08162 USD.'